In [1]:
# CELL 1 — Environment setup + Hugging Face token for gated models
!pip install -q transformers accelerate torch torchvision pillow tqdm sentencepiece

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
print("✅ HF Token loaded securely.")


✅ HF Token loaded securely.


In [3]:
# ====================================================
# CELL 1: SETUP & MODEL LOADING (SPECIALIZED MODELS)
# ====================================================

# 1. INSTALL LIBRARIES
# We need sentence-transformers for the BGE text model
!pip install sentence-transformers transformers torch pandas tqdm accelerate -q

# 💡 IMPORTANT: You may need to restart the kernel after installation
# for the new libraries to be loaded correctly. (In Kaggle: Run > Restart Session)

import os
import gc
import torch
import requests
import numpy as np
import pandas as pd
from io import BytesIO
from PIL import Image
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoProcessor, SiglipVisionModel
from sentence_transformers import SentenceTransformer

# ---------- CONFIGURATION ----------
# !! IMPORTANT: Set your CSV file path here
csv_path = "/kaggle/input/train-csv/shard_0.csv"
output_path = "/kaggle/working/specialized_embeddings1.npz"
device = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64 # Can use a larger batch size with these smaller models

# ---------- LOAD SPECIALIZED TEXT MODEL ----------
print("Loading specialized text model (BAAI/bge-small-en-v1.5)...")
text_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
print("Text model loaded successfully.")

# ---------- LOAD SPECIALIZED IMAGE MODEL ----------
print("Loading specialized image model (google/siglip-base-patch16-224)...")
image_model_name = "google/siglip-base-patch16-224"
image_processor = AutoProcessor.from_pretrained(image_model_name)
image_model = SiglipVisionModel.from_pretrained(
    image_model_name,
    torch_dtype=torch.float16
).to(device).eval()
print("Image model loaded successfully.")


# ---------- LOAD DATA & CACHE IMAGES ----------
df = pd.read_csv(csv_path)
image_url_column = 'image_link'
if image_url_column not in df.columns:
    raise ValueError(f"Column '{image_url_column}' not found. Please update the variable.")
    
print(f"Loaded shard: {len(df)} rows")
IMG_DIR = "/kaggle/working/img_cache"
os.makedirs(IMG_DIR, exist_ok=True)

def fetch_image(index, url):
    path = os.path.join(IMG_DIR, f"{index}.jpg")
    if os.path.exists(path): return
    try:
        if isinstance(url, str) and url.strip():
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            Image.open(BytesIO(r.content)).convert("RGB").save(path)
    except Exception: pass

print("Caching images from URLs...")
with ThreadPoolExecutor(max_workers=16) as executor:
    list(tqdm(executor.map(fetch_image, df.index, df[image_url_column]), total=len(df)))

print("\n✅ Setup complete.")

Loading specialized text model (BAAI/bge-small-en-v1.5)...
Text model loaded successfully.
Loading specialized image model (google/siglip-base-patch16-224)...
Image model loaded successfully.
Loaded shard: 7000 rows
Caching images from URLs...


100%|██████████| 7000/7000 [05:26<00:00, 21.46it/s]


✅ Setup complete.


In [4]:
# ====================================================
# CELL 2: GENERATE TEXT EMBEDDINGS
# ====================================================

# !! IMPORTANT: Replace 'catalog_content' if your text column has a different name
text_column = 'catalog_content'
if text_column not in df.columns:
    raise ValueError(f"Column '{text_column}' not found. Please update the variable.")

print("Generating text embeddings...")
# Convert text column to a list, handling any missing values
texts_to_embed = df[text_column].fillna('').astype(str).tolist()

# The sentence-transformers library handles tokenization, padding, and pooling automatically.
# We also normalize the embeddings, which is a best practice for similarity tasks.
text_embeddings = text_model.encode(
    texts_to_embed,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print(f"\n✅ Text embeddings generated. Shape: {text_embeddings.shape}")

Generating text embeddings...


Batches:   0%|          | 0/110 [00:00<?, ?it/s]


✅ Text embeddings generated. Shape: (7000, 384)


In [5]:
# ====================================================
# CELL 3: GENERATE IMAGE EMBEDDINGS & SAVE
# ====================================================

def load_or_blank(p):
    try: return Image.open(p).convert("RGB")
    except Exception: return Image.new("RGB", (224, 224))

all_image_embeddings = []
print("Generating image embeddings...")
# Process the cached images in batches
for i in tqdm(range(0, len(df), BATCH_SIZE), desc="Image batches"):
    batch_df = df.iloc[i:i + BATCH_SIZE]
    
    batch_paths = [os.path.join(IMG_DIR, f"{idx}.jpg") for idx in batch_df.index]
    batch_images = [load_or_blank(p) for p in batch_paths]
    
    inputs = image_processor(images=batch_images, return_tensors="pt").to(device)

    with torch.no_grad():
        # The SigLIP model's embedding is the pooler_output
        outputs = image_model(**inputs)
        # We normalize the image embeddings as well for consistency
        image_batch_embeddings = torch.nn.functional.normalize(outputs.pooler_output, dim=-1)
        
    all_image_embeddings.append(image_batch_embeddings.cpu().numpy())

# Stack the batches into a single array
image_embeddings = np.vstack(all_image_embeddings)
print(f"\n✅ Image embeddings generated. Shape: {image_embeddings.shape}")

# ---------- SAVE COMBINED EMBEDDINGS ----------
# !! IMPORTANT: Replace 'price' if your price column has a different name
price_column = 'price'
np.savez_compressed(output_path,
    ids=df.index.to_numpy(),
    text=text_embeddings.astype(np.float16),
    image=image_embeddings.astype(np.float16),
    price=df[price_column].astype(np.float32).to_numpy()
)
print(f"✅ All embeddings combined and saved to: {output_path}")

Generating image embeddings...


Image batches: 100%|██████████| 110/110 [06:24<00:00,  3.49s/it]



✅ Image embeddings generated. Shape: (7000, 768)
✅ All embeddings combined and saved to: /kaggle/working/specialized_embeddings.npz
